# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library. All entities (record sets, fields, columns) are referenced by their `@id` as defined in the Croissant schema.

### Dataset Source
Croissant schema JSON-LD URL:
```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the Croissant dataset and display key metadata attributes using the mlcroissant library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

meta = dataset.metadata  # Single object, not a mapping or sequence
print(f"Name: {getattr(meta, 'name', 'N/A')}")
print(f"Description: {getattr(meta, 'description', 'N/A')}")
print(f"Published: {getattr(meta, 'date_published', 'N/A')}")
print(f"License: {getattr(meta, 'license', 'N/A')}")
print(f"Version: {getattr(meta, 'version', 'N/A')}")

## 2. Data Overview
List available record sets, fields, and their `@id`s for exploration.

In [ ]:
# Find available record sets and show their @id, name, and description if present

from mlcroissant.types import RecordSet, Field
record_sets: list = getattr(meta, 'record_sets', None)

if record_sets is None or not record_sets:
    print("No record sets found via .record_sets; attempting .record_set attribute...")
    # Some datasets use singular 'record_set'
    record_sets = getattr(meta, 'record_set', [])

if not record_sets:
    raise ValueError('No record sets found in metadata!')

print("Available Record Sets:")
all_record_set_ids = []
for rs in record_sets:
    if isinstance(rs, RecordSet):
        rid = getattr(rs, '@id', None) or getattr(rs, 'id_', None)
        print(f"- RecordSet @id: {rid} | name: {getattr(rs, 'name', '')} | description: {getattr(rs, 'description', '')}")
        all_record_set_ids.append(rid)
        # Print fields for this record set
        if hasattr(rs, 'fields'):
            print("     Fields:")
            for f in getattr(rs, 'fields', []):
                if isinstance(f, Field):
                    print(f"      - Field @id: {getattr(f, '@id', getattr(f, 'id_', ''))} | name: {getattr(f, 'name', '')}")
    elif isinstance(rs, dict):
        rid = rs.get('@id', None)
        print(f"- RecordSet @id: {rid} (dict)")
        all_record_set_ids.append(rid)
    else:
        # Usually string @id
        print(f"- RecordSet: {rs}")
        all_record_set_ids.append(str(rs))

# For demonstration, print a small sample of records from the first record set
record_set_to_show = all_record_set_ids[0]
print(f"\nSample records from RecordSet @id: {record_set_to_show}")
for i, rec in enumerate(dataset.records(record_set=record_set_to_show)):
    print(rec)
    if i >= 2:
        break

## 3. Data Extraction
Extract all data from the main record set(s) to a pandas DataFrame using their `@id`s. You will need to use the exact `@id` of the record sets and fields as shown in the previous section.

In [ ]:
# List all record set @ids for analysis
record_sets_ids = all_record_set_ids  # As determined above
dataframes = {}

for rid in record_sets_ids:
    records = list(dataset.records(record_set=rid))
    if len(records):
        df = pd.DataFrame(records)
    else:
        df = pd.DataFrame()
    dataframes[rid] = df

# Show columns of the first record set
main_rs_id = record_sets_ids[0]
print(f"Columns for RecordSet '@id': {main_rs_id}")
print(dataframes[main_rs_id].columns.tolist())
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Demonstrate filtering, normalization, and (optionally) grouping for numeric fields. Please replace `<numeric_field_id>` and `<group_field>` with the actual field names (usually match column names in dataframe that are mapped to the `@id` of each field).

**Note:** All manipulations use the original Croissant field `@id` (or its mapped column name if different).

In [ ]:
# Pick a numeric field to analyze. Replace as appropriate.

df = dataframes[main_rs_id]
print("Detecting numeric fields available in the main DataFrame:")
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
print(numeric_columns)
if not numeric_columns:
    # Attempt to convert float-like string columns
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            pass
    numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()

if not numeric_columns:
    print("No numeric columns available for analysis.")
else:
    numeric_field = numeric_columns[0]
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with '{numeric_field}' > mean:")
    print(filtered_df.head())

    # Normalize this field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a likely categorical field (if any)
    candidate_cats = df.select_dtypes(include=['object']).columns.tolist()
    group_field = None
    for col in candidate_cats:
        if df[col].nunique() > 1 and df[col].nunique() < 50:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped data by '{group_field}':")
        print(grouped_df.head())

## 5. Visualization
Plot a histogram of the numeric field and (optionally) boxplots or barplots by group if available.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if not numeric_columns:
    print("No numeric columns for visualization.")
else:
    field = numeric_field
    plt.figure(figsize=(7, 4))
    df[field].hist(bins=30)
    plt.title(f"Distribution of {field}")
    plt.xlabel(field)
    plt.ylabel("Count")
    plt.grid(True, alpha=0.3)
    plt.show()
    
    if group_field:
        plt.figure(figsize=(8,4))
        filtered_df.boxplot(column=field, by=group_field)
        plt.title(f"{field} by {group_field}")
        plt.suptitle("")
        plt.ylabel(field)
        plt.xlabel(group_field)
        plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded FAIR² protein mass spectrometry data as described by its Croissant schema
- Explored available record sets and fields using their `@id`s
- Extracted dataframes for analysis, filtering, normalization, and visualization

This structure provides a robust template for future FAIR dataset explorations via Croissant. For more advanced studies (ML, statistical tests), you may now continue building upon these loaded dataframes!